# 🇲🇬 Serveur de Traduction EN/FR vers Malgache (Google Colab)

Ce notebook lance le backend FastAPI sur **Google Colab avec GPU gratuit (T4)** et expose l'API publiquement via **ngrok**.

### 🔑 Configuration préalable des Secrets dans Google Colab :
Dans le volet vertical à gauche de Colab, cliquez sur l'icône **Clé 🔑 (Secrets)** et ajoutez **2 secrets** :
1. **`NGROK_AUTHTOKEN`** : Votre token gratuit copié depuis [dashboard.ngrok.com](https://dashboard.ngrok.com).
2. **`API_KEY`** : La clé secrète de votre choix pour sécuriser l'accès à votre API (ex: `mon_super_secret_123`).

*(N'oubliez pas d'activer l'interrupteur **"Accès au notebook"** pour chacun de ces deux secrets !)*

### ⚡ Activer le GPU :
Dans le menu : **Exécution** > **Changer le type d'exécution** > sélectionnez **GPU T4** > **Enregistrer**.

In [ ]:
# Récupération du dépôt GitHub
REPO_URL = "https://github.com/franck504/en-fr-to-mg-api.git"
BRANCH = "main"
PROJECT_DIR = "/content/en_fr_to_MG"

import os
import subprocess

if os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print(f"Projet prêt dans {PROJECT_DIR}")

In [ ]:
# Installation des dépendances légères (PyTorch avec CUDA est déjà pré-installé sur Colab)
!pip install -q -U -r requirements-colab.txt

In [ ]:
import os
from google.colab import userdata

# 🔒 Récupération sécurisée du token ngrok depuis Colab Secrets
try:
    os.environ["NGROK_AUTHTOKEN"] = userdata.get('NGROK_AUTHTOKEN')
    print("✅ Secret NGROK_AUTHTOKEN chargé !")
except Exception as exc:
    raise RuntimeError(
        "Veuillez ajouter le secret 'NGROK_AUTHTOKEN' dans le volet Secrets (icône clé 🔑 à gauche). "
        "Obtenez ce token gratuit sur https://dashboard.ngrok.com"
    ) from exc

# 🔒 Récupération de la clé API pour protéger le serveur
try:
    os.environ["API_KEY"] = userdata.get('API_KEY')
    print("✅ Secret API_KEY chargé avec succès !")
except Exception as exc:
    raise RuntimeError(
        "Veuillez ajouter le secret 'API_KEY' dans le volet Secrets (icône clé 🔑 à gauche) avec la clé de votre choix."
    ) from exc

# Configuration du serveur local
os.environ["ENABLE_NGROK"] = "true"
os.environ["HOST"] = "0.0.0.0"
os.environ["PORT"] = "8000"
os.environ["PROVIDER"] = "hf_seq2seq"
os.environ["HF_MODEL_NAME"] = "facebook/nllb-200-distilled-600M"
os.environ["HF_MODEL_FAMILY"] = "nllb"
os.environ["HF_DEVICE"] = "auto"
os.environ["MODEL_CACHE_DIR"] = "/content/hf_models"
os.environ["LOAD_MODEL_ON_STARTUP"] = "false"

print("🚀 Configuration validée pour Google Colab.")

In [ ]:
# Démarrage du serveur et ouverture du tunnel ngrok public
!python run_colab.py

## 💡 Utilisation de votre API sécurisée

Remplacez `VOTRE_CLE_API` par la valeur que vous avez mise dans le secret `API_KEY`.

### Exemple Python avec SDK OpenAI :
```python
from openai import OpenAI

client = OpenAI(
    base_url="https://votre-id.ngrok-free.app/v1",
    api_key="VOTRE_CLE_API", # La clé définie dans Colab Secrets
)
response = client.chat.completions.create(
    model="nllb",
    messages=[{"role": "user", "content": "Hello, how are you?"}],
)
print(response.choices[0].message.content)
```

### Exemple cURL direct :
```bash
curl -X POST https://votre-id.ngrok-free.app/translate \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer VOTRE_CLE_API" \
  -d '{"text": "Bonjour", "source_lang": "auto", "target_lang": "mg"}'
```